In [5]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout


In [6]:

# Load only the intended climate/material simulation datasets

from pathlib import Path
import pandas as pd

repo_dir = Path.cwd().resolve()

if repo_dir.name == "Final Notebooks":
    repo_dir = repo_dir.parent

data_dir = repo_dir / "Data"

cities = [
    "Gatwick",
    "Barcelona",
    "Norway",
    "Dubai",
    "Singapore",
    "Bath"
]

expected_files = [
    f"{city}{suffix}.csv"
    for city in cities
    for suffix in ["", "02", "03"]
    if not (city == "Bath" and suffix == "")
]

dfs = []

for filename in expected_files:
    filepath = data_dir / filename

    if not filepath.exists():
        raise FileNotFoundError(f"Missing dataset: {filepath}")

    df = pd.read_csv(filepath)
    df["City"] = filepath.stem
    df["Datetime"] = pd.to_datetime(
    df["Datetime"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)

data = data.sort_values(
    ["City", "Datetime"]
).reset_index(drop=True)

print("Simulation files loaded:", len(dfs))
print("Combined data shape:", data.shape)



Simulation files loaded: 17
Combined data shape: (148920, 6)


In [7]:

# Cell 3: Feature engineering
# Calculate historical features separately for each simulation.

import numpy as np
import pandas as pd

data["Datetime"] = pd.to_datetime(
    data["Datetime"], dayfirst=True, errors="coerce"
)

data = data.dropna(subset=["Datetime"])
data = data.sort_values(["City", "Datetime"]).copy()

# Time features
data["hour"] = data["Datetime"].dt.hour
data["dayofyear"] = data["Datetime"].dt.day_of_year

data["hour_sin"] = np.sin(2 * np.pi * data["hour"] / 24)
data["hour_cos"] = np.cos(2 * np.pi * data["hour"] / 24)
data["day_sin"] = np.sin(2 * np.pi * data["dayofyear"] / 365)
data["day_cos"] = np.cos(2 * np.pi * data["dayofyear"] / 365)

# Group by individual climate/material simulation
groups = data.groupby("City", sort=False)

# Historical lag features
for lag in [1, 2, 3, 6, 12, 24]:
    for column, prefix in [
        ("Outdoor_Temp", "outdoor_temp"),
        ("Indoor_Temp", "indoor_temp"),
        ("Outdoor_Humidity", "outdoor_hum"),
        ("Indoor_Humidity", "indoor_hum")
    ]:
        data[f"{prefix}_lag_{lag}"] = groups[column].shift(lag)

# Rolling averages based only on previous observations
for window in [3, 6, 12, 24]:
    for column, prefix in [
        ("Outdoor_Temp", "outdoor_temp"),
        ("Indoor_Temp", "indoor_temp"),
        ("Outdoor_Humidity", "outdoor_hum"),
        ("Indoor_Humidity", "indoor_hum")
    ]:
        data[f"{prefix}_roll_{window}"] = (
            data.groupby("City")[column]
            .transform(
                lambda s: s.shift(1).rolling(window).mean()
            )
        )

# Remove rows with insufficient historical observations
data = data.dropna().reset_index(drop=True)

print("Feature engineering complete.")
print("Processed data shape:", data.shape)
print("Number of simulations:", data["City"].nunique())
print("Missing values:", data.isna().sum().sum())


Feature engineering complete.
Processed data shape: (148512, 52)
Number of simulations: 17
Missing values: 0


In [9]:

# Cell 4: Prepare data and create LSTM sequences
# Chronological 70/15/15 split within each simulation

from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

seq_length = 48

target_cols = ["Indoor_Temp", "Indoor_Humidity"]
feature_cols = [
    c for c in data.columns
    if c not in ["Datetime", "City", *target_cols]
]

# Separate each simulation chronologically
splits = {"train": [], "val": [], "test": []}

for city, group in data.groupby("City"):
    group = group.sort_values("Datetime").copy()

    n = len(group)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    splits["train"].append(group.iloc[:train_end])
    splits["val"].append(group.iloc[train_end:val_end])
    splits["test"].append(group.iloc[val_end:])

# Fit scalers using training observations only
training_data = pd.concat(splits["train"], ignore_index=True)

scaler_X = StandardScaler()
scaler_y = StandardScaler()

scaler_X.fit(training_data[feature_cols].astype(float))
scaler_y.fit(training_data[target_cols].astype(float))

def create_grouped_sequences(groups, seq_length):
    X_all, y_all = [], []

    for group in groups:
        X = scaler_X.transform(
            group[feature_cols].astype(float)
        )
        y = scaler_y.transform(
            group[target_cols].astype(float)
        )

        for i in range(seq_length, len(group)):
            X_all.append(X[i-seq_length:i])
            y_all.append(y[i])

    return (
        np.asarray(X_all, dtype=np.float32),
        np.asarray(y_all, dtype=np.float32)
    )

X_train_seq, y_train_seq = create_grouped_sequences(
    splits["train"], seq_length
)

X_val_seq, y_val_seq = create_grouped_sequences(
    splits["val"], seq_length
)

X_test_seq, y_test_seq = create_grouped_sequences(
    splits["test"], seq_length
)

print("Training:", X_train_seq.shape, y_train_seq.shape)
print("Validation:", X_val_seq.shape, y_val_seq.shape)
print("Testing:", X_test_seq.shape, y_test_seq.shape)
print("Missing values:", np.isnan(X_train_seq).sum())


Training: (103139, 48, 48) (103139, 2)
Validation: (21454, 48, 48) (21454, 2)
Testing: (21471, 48, 48) (21471, 2)
Missing values: 0


In [ ]:
# Cell 5: Define & train LSTM model with sequences

model = Sequential()
model.add(LSTM(64, activation='tanh', input_shape=(seq_length, X_train_seq.shape[2])))
model.add(Dropout(0.2))
model.add(Dense(32, activation='relu'))
model.add(Dense(2))  # two outputs: Indoor_Temp & Indoor_Humidity

model.compile(optimizer='adam', loss='mse')

history = model.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=20,
    batch_size=32,
    verbose=1
)


In [ ]:
# Cell 6: Evaluation with sequence inputs

y_pred_scaled = model.predict(X_test_seq)
y_pred = scaler_y.inverse_transform(y_pred_scaled)
y_test_inv = scaler_y.inverse_transform(y_test_seq)

target = ['Indoor_Temp', 'Indoor_Humidity']
for i, col in enumerate(target):
    mae = mean_absolute_error(y_test_inv[:, i], y_pred[:, i])
    rmse = np.sqrt(mean_squared_error(y_test_inv[:, i], y_pred[:, i]))
    r2 = r2_score(y_test_inv[:, i], y_pred[:, i])
    print(f"{col} - MAE: {mae:.3f}, RMSE: {rmse:.3f}, R²: {r2:.3f}")
